# Workstream 2: LitCoin Embedding And Retrieval Comparison

This notebook compares OpenAI and PubMedBERT SapBERT relationship embeddings for the same LitCoin relationship edges. It uses tested helper functions from `analysis.embedding_comparison` so that validation, projections, metrics, and exports are reproducible outside notebook cell state.

Projection caveat: OpenAI and SapBERT embeddings occupy different vector spaces and may have different dimensions. The panels below are fitted separately. Compare highlighted edge membership, nearest-neighbor agreement, publication concentration, and predicate/category patterns, not absolute axis direction, rotation, or inter-panel coordinates.

In [ ]:
from pathlib import Path

from analysis.embedding_comparison import (
    build_embedding_collection,
    build_retrieval_comparison_table,
    comparison_manifest,
    create_projection_figure,
    deduplicate_exact_duplicate_rows,
    match_embedding_collections,
    nearest_neighbor_jaccard,
    nearest_neighbors,
    project_embeddings,
    read_jsonl_rows,
    same_metadata_fraction,
    summarize_anchor_diversity,
    write_json,
    write_projection_html,
    write_rows_csv,
)

PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / "scripts" / "data"
OUTPUT_DIR = PROJECT_ROOT / "analysis" / "outputs" / "embedding_comparison"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OPENAI_RELATIONSHIP_EXPORT = DATA_DIR / "relationship_embeddings.jsonl"
SAPBERT_RELATIONSHIP_EXPORT = DATA_DIR / "sapbert_relationship_embeddings.jsonl"
RANDOM_SEED = 13
TOP_K = 10

QUERIES = [
    "drug resistance in cancer",
    "genes involved in chemoresistance in cancer",
    "PTEN cancer chemoresistance",
    "therapeutic response relationships",
]

## Load And Validate Exports

The JSONL exports should contain one row per relationship per model. The strict loader fails on missing IDs, duplicate IDs within one model export, and inconsistent dimensions. The current local exports contain exact duplicate relationship rows from an older undirected dump query, so this notebook removes only exact duplicate rows and records that normalization in the manifest. Embedding arrays are retained in memory only and are removed from exported tables and figure hover payloads.

In [ ]:
openai_rows, openai_normalization = deduplicate_exact_duplicate_rows(
    read_jsonl_rows(OPENAI_RELATIONSHIP_EXPORT), model_name="openai"
)
sapbert_rows, sapbert_normalization = deduplicate_exact_duplicate_rows(
    read_jsonl_rows(SAPBERT_RELATIONSHIP_EXPORT), model_name="sapbert"
)
openai_edges = build_embedding_collection(openai_rows, model_name="openai", embedding_key="embedding")
sapbert_edges = build_embedding_collection(sapbert_rows, model_name="sapbert", embedding_key="sapbert_embedding")
matched = match_embedding_collections(openai_edges, sapbert_edges)

manifest = comparison_manifest(
    collections=[openai_edges, sapbert_edges],
    projection_parameters={"method": "pca", "random_seed": RANDOM_SEED},
    queries=QUERIES,
    seed=RANDOM_SEED,
)
manifest["coverage"] = matched.coverage_report
manifest["input_normalization"] = {"openai": openai_normalization, "sapbert": sapbert_normalization}
write_json(manifest, OUTPUT_DIR / "manifest.json")
matched.coverage_report

## Side-By-Side Projections

PCA is used here as a deterministic reference projection. UMAP can be requested with `method="umap"` if `umap-learn` is installed, but it should remain an exploratory local-neighborhood view.

In [ ]:
openai_projection = project_embeddings(openai_edges, relationship_ids=matched.relationship_ids, method="pca", seed=RANDOM_SEED)
sapbert_projection = project_embeddings(sapbert_edges, relationship_ids=matched.relationship_ids, method="pca", seed=RANDOM_SEED)

projection_rows = openai_projection.rows + sapbert_projection.rows
write_rows_csv(projection_rows, OUTPUT_DIR / "pca_projection_rows.csv")
write_projection_html([openai_projection, sapbert_projection], OUTPUT_DIR / "pca_projection.html", color_field="predicate_family")

create_projection_figure([openai_projection, sapbert_projection], color_field="predicate_family")

## Neighborhood Agreement

These metrics compare local neighborhood membership within each embedding model. They do not compare raw coordinate values across model spaces.

In [ ]:
openai_neighbors = nearest_neighbors(openai_edges, k=TOP_K, relationship_ids=matched.relationship_ids)
sapbert_neighbors = nearest_neighbors(sapbert_edges, k=TOP_K, relationship_ids=matched.relationship_ids)

neighbor_metrics = {
    "top_k": TOP_K,
    "nearest_neighbor_jaccard": nearest_neighbor_jaccard(openai_neighbors, sapbert_neighbors, k=TOP_K),
    "openai_same_publication_fraction": same_metadata_fraction(openai_edges, openai_neighbors, metadata_field="publication_id", k=TOP_K),
    "sapbert_same_publication_fraction": same_metadata_fraction(sapbert_edges, sapbert_neighbors, metadata_field="publication_id", k=TOP_K),
    "openai_same_predicate_family_fraction": same_metadata_fraction(openai_edges, openai_neighbors, metadata_field="predicate_family", k=TOP_K),
    "sapbert_same_predicate_family_fraction": same_metadata_fraction(sapbert_edges, sapbert_neighbors, metadata_field="predicate_family", k=TOP_K),
}
write_json(neighbor_metrics, OUTPUT_DIR / "nearest_neighbor_metrics.json")
neighbor_metrics

## Retrieval Result Comparison

Populate `retrieval_result_sets` from saved search diagnostics or scripted offline retrieval runs. Keep raw dense, keyword, hybrid, and final reranked/diversified anchors as separate result sets so the table can distinguish candidate generation from final anchor selection.

In [ ]:
retrieval_result_sets = []  # Fill with RetrievalResultSet(...) values from saved diagnostics.
retrieval_table = build_retrieval_comparison_table(retrieval_result_sets)
write_rows_csv(retrieval_table, OUTPUT_DIR / "retrieval_comparison.csv")

diversity_rows = [summarize_anchor_diversity([row], query=row["query"]) for row in retrieval_table]
write_rows_csv(diversity_rows, OUTPUT_DIR / "anchor_diversity_summary.csv")
retrieval_table[:5]

## Conclusions Template

Direct observations:

- Record matched edge coverage, dimensions, nearest-neighbor agreement, and retrieval overlaps from the generated tables.
- Identify whether top results concentrate in a small number of publications or predicate families.
- Compare raw dense candidates with reranked/diversified anchors separately.

Interpretations:

- Explain whether differences appear useful for graph exploration entry points rather than claiming global model superiority.
- Connect examples to the path-centric workflow: Search -> Candidate Paths -> Human Selection -> Expansion.

Future hypotheses:

- Candidate retrieval may benefit from model-specific routing or hybrid weighting, but only after evaluating more queries and human judgments.
- A small diagnostic panel may be useful later if it explains selected anchors without turning the Dash app into an embedding browser.